# CV runner — chạy 5-fold cho MỘT config bất kỳ

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Thay cho `07_e4_cv_folds.ipynb`, vốn khoá cứng vào `baseline_3dpatch.yaml` và còn dùng
logic dò đường dẫn cũ đã sai (`/kaggle/input/<slug>` thay vì
`/kaggle/input/datasets/<user>/<slug>` — WORKLOG S-084). Notebook này nhận **tên config**
làm tham số, và dùng lại đúng khối dò đã sửa của notebook 08.

**Thí nghiệm đang chạy: E5 — focal loss.** `configs/e5_focal.yaml` khác
`configs/baseline_3dpatch.yaml` đúng ba chỗ: `loss.name`, `loss.gamma`, `output_dir`.
Cùng cache, cùng seed 1337, cùng 5 fold, cùng mọi hyperparam khác — nên chênh lệch
quy được cho đúng một biến.

**Hai giả thuyết** (chi tiết ở đầu `configs/e5_focal.yaml`):
- **H1** macro-F1 out-of-fold > 0.6851 của E4 *(CGHNet Bảng 4: Focal 81.8 vs CE 79.9)*
- **H2** ECE chưa hiệu chỉnh < 0.2030 của E4 *(Mukhoti và cs. 2020)* — **quan trọng hơn H1**
  với dự án này, vì đóng góp headline là trustworthiness.

**Ngân sách:** ~3.75h/fold × 300 epoch. Session Kaggle 12h ⇒ **2 fold mỗi session**,
ba session là xong 5 fold. `resume: true` nên bị ngắt thì chạy lại chính cell train.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- HAI THAM SỐ DUY NHẤT CẦN SỬA ------------------------------------------
CONFIG_NAME = "e5_focal.yaml"     # tên file trong configs/
FOLDS = [1, 2]                    # session sau: [3, 4] rồi [5]
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

EXPERIMENT = Path(CONFIG_NAME).stem
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
assert CFG_PATH.exists(), f"không thấy {CFG_PATH}"
CFG = load_yaml(CFG_PATH)

print(f"\nthí nghiệm: {EXPERIMENT} · fold {FOLDS}")
print(f"loss: {CFG['loss']}")
print(f"output: {os.environ['LLDMMRI_OUTPUT_DIR']}")

## Cổng 0 ⚠️ — config này khác baseline ở ĐÚNG những chỗ nào

In ra diff để thấy tận mắt, thay vì tin rằng chỉ có một biến đổi. Một so sánh có kiểm
soát mà lỡ đổi hai biến thì không quy kết được nguyên nhân, và điều đó **không tự lộ ra**
ở bất kỳ đâu trong kết quả.

In [ ]:
BASE = load_yaml(REPO / "configs" / "baseline_3dpatch.yaml")


def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flatten(v, prefix + k + "."))
        else:
            out[prefix + k] = v
    return out


fa, fb = flatten(BASE), flatten(CFG)
diff = {k: (fa.get(k), fb.get(k)) for k in sorted(set(fa) | set(fb)) if str(fa.get(k)) != str(fb.get(k))}

print(f"{CONFIG_NAME} khác baseline_3dpatch.yaml ở {len(diff)} khoá:")
for k, (a, b) in diff.items():
    print(f"  {k}: {a!r} -> {b!r}")

khac_ngoai_loss = [k for k in diff if not k.startswith("loss.") and k not in ("output_dir", "fold")]
if khac_ngoai_loss:
    print(f"\n⚠ CÓ khác biệt NGOÀI khối loss: {khac_ngoai_loss}")
    print("  Nếu không cố ý thì dừng lại — đang đổi nhiều hơn một biến.")
else:
    print("\n✓ chỉ khác ở khối loss (+ output_dir) — so sánh có kiểm soát")

## 1. Cache

Nhận diện cache bằng **nội dung** `cache_meta.json`, không bằng tên dataset. Xem
WORKLOG S-082..S-086 về lý do (bốn lần đoán sai đường dẫn liên tiếp).

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

# Cache E4 nhận diện bằng NỘI DUNG `cache_meta.json`, không bằng tên dataset. Tên do
# người upload đặt và đã lệch một lần rồi (`lld-mmri-lesion-tight/cache_lesion_tight`
# chứ không phải `lld-mmri-e4-per-phase` như đoán ở S-080). Ba khoá này là thứ phân
# biệt E4 với mọi cache trước đó.
E4_KEYS = {
    "align_phases": "per_phase",          # <- phân biệt E4 với E3
    "target_size": [112, 112, 32],        # <- phân biệt E3/E4 với E0/E1
    "crop_mode": "lesion_tight",          # <- phân biệt E1+ với E0
}

# ---------------------------------------------------------------------------
# KHÔNG hardcode độ sâu. Kaggle mount ở `/kaggle/input/datasets/<user>/<slug>/...`
# chứ không phải `/kaggle/input/<slug>/...` như mọi notebook trước giả định, và độ
# sâu đó có thể đổi tiếp. Dò theo TÊN FILE mốc, sâu bao nhiêu cũng thấy. Đây là lần
# thứ tư sửa cùng một lớp lỗi (S-081 → S-084); nguyên nhân gốc luôn là một giả định
# về hình dạng đường dẫn.
#
# MỘT lượt `os.walk` duy nhất thu hết mọi thứ cần. Không dùng nhiều `rglob` riêng:
# dataset gốc là 83.7GB / ~4000 file trên ổ mạng, và mỗi `rglob` là một lượt duyệt
# toàn cây — 11 lượt thì chờ rất lâu mà chẳng được gì thêm.
# ---------------------------------------------------------------------------
import json as _json
import os as _os
import re as _re

_cfg_data = load_yaml(REPO / "configs" / "data.yaml")
_ann_name = Path(_cfg_data["annotation_rel"]).name

interesting = {}       # thư mục -> số .npz/.pt/meta, để in bảng chẩn đoán
meta_paths = []        # cache_meta.json tìm được
_ann = []              # file annotation của dữ liệu gốc

for dirpath, dirnames, filenames in _os.walk(INPUT_ROOT):
    dirnames[:] = [x for x in dirnames if x not in (".cache", ".git")]  # rác tải HF
    here = {"npz": 0, "pt": 0, "meta": 0}
    for name in filenames:
        full = Path(dirpath) / name
        if name.endswith(".npz"):
            here["npz"] += 1
        elif name.endswith(".pt"):
            here["pt"] += 1
        elif name == "cache_meta.json":
            here["meta"] += 1
            meta_paths.append(full)
        elif name == _ann_name:
            _ann.append(full)
    if any(here.values()):
        interesting[Path(dirpath)] = here


def read_caches(paths):
    out = []
    for p in sorted(paths):
        try:
            out.append((p.parent, _json.loads(p.read_text("utf-8"))))
        except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo, không nuốt lỗi thật
            out.append((p.parent, {"__loi__": repr(exc)}))
    return out


def matches_e4(meta):
    return all(meta.get(k) == v for k, v in E4_KEYS.items())


print(f"=== Thư mục có dữ liệu dưới {INPUT_ROOT} ===")
for d in sorted(interesting)[:25]:
    c = interesting[d]
    print(f"  {d}\n      {' · '.join(f'{c[k]} {k}' for k in ('npz', 'pt', 'meta') if c[k])}")
if not interesting:
    print("  (trống — chưa mount dataset nào có .npz/.pt)")

print(f"\n=== Dữ liệu gốc ({_ann_name}) ===")
for p in sorted(_ann)[:5]:
    print(f"  ✓ {p.parent.parent}")
if not _ann:
    print("  KHÔNG thấy — chỉ cần nếu phải build cache (xem ngay dưới)")

caches = read_caches(meta_paths)
print(f"\n=== {len(caches)} cache có cache_meta.json ===")
for path, meta in caches:
    mark = "✓ E4" if matches_e4(meta) else "  --"
    print(
        f"  {mark}  {path}\n"
        f"        crop={meta.get('crop_mode')} size={meta.get('target_size')} "
        f"align={meta.get('align_phases')}"
    )

e4 = [p for p, m in caches if matches_e4(m)]
if e4:
    CACHE_DIR = e4[0]
    BUILD_NEEDED = False
else:
    # Thư mục có nhiều .npz nhưng KHÔNG có meta: không dùng được, và phải nói rõ vì sao.
    # Hình dạng mảng cho biết target_size, nhưng KHÔNG cho biết `align_phases` —
    # E3 (reference) và E4 (per_phase) có cùng shape [8,112,112,32]. Nhận nhầm E3
    # thành E4 sẽ cho ra một bảng kết quả sai mà trông hoàn toàn hợp lý.
    for d, c in sorted(interesting.items()):
        if c["npz"] > 100 and not c["meta"]:
            print(f"\n⚠ {d} có {c['npz']} file .npz nhưng KHÔNG có cache_meta.json.")
            print("  Không dùng được: shape cho biết target_size nhưng KHÔNG phân biệt được")
            print("  E3 (align=reference) với E4 (align=per_phase) — hai cái cùng shape.")
    BUILD_NEEDED = True
    CACHE_DIR = Path("/kaggle/working/cache_e4")
    if _ann:
        print("\n=> sẽ BUILD lại cache E4 (~26 phút). Dữ liệu gốc đã có ✓")
    else:
        print(
            "\n=> CẦN BUILD cache E4 nhưng CHƯA MOUNT dữ liệu gốc.\n"
            f"   Mount dataset chứa {_cfg_data['annotation_rel']} "
            f"(ứng viên: {_cfg_data.get('data_root_candidates')}),\n"
            "   rồi chạy lại từ cell này."
        )
print("\ncache:      ", CACHE_DIR, "(CHƯA CÓ — sẽ build ở cell dưới)" if BUILD_NEEDED else "")

## 1b. Build cache nếu chưa có (~26 phút)

Cache E4 chưa từng được lưu thành Kaggle Dataset nên nhánh này thường sẽ chạy. Cân
nhắc "Save Version" giữ `/kaggle/working/cache_e4` để lần sau khỏi mất 26 phút.

In [ ]:
if BUILD_NEEDED:
    from src.utils.io import resolve_data_root

    # Data root khai ở configs/data.yaml, KHÔNG ở preprocess_*.yaml — file preprocess
    # chỉ có tham số tiền xử lý. `build_cache` cũng đọc data.yaml (xem hàm main của
    # nó). Kiểm trước ở đây chỉ để fail nhanh, thay vì chết giữa job 26 phút.
    cfg_data = load_yaml(REPO / "configs" / "data.yaml")
    try:
        data_root = resolve_data_root(cfg_data)
    except Exception as exc:
        # RuntimeError chứ không SystemExit: SystemExit làm IPython lỗi khi dựng
        # traceback và che mất thông báo thật bằng một trang lỗi của chính nó.
        data_root, exc_msg = None, str(exc)
    else:
        exc_msg = None

    # `resolve_data_root` trả về `config['data_root']` mà KHÔNG xác minh khi mọi cách
    # dò đều trượt (src/utils/io.py:219-225). Trên Kaggle nó sẽ là `data/lldmmridataset`
    # tương đối, không tồn tại — và job 26 phút sẽ chết giữa chừng. Xác minh ở đây.
    ann = (data_root / cfg_data["annotation_rel"]) if data_root else None
    if ann is None or not ann.exists():
        raise RuntimeError(
            f"Không tìm thấy dữ liệu LLD-MMRI gốc.\n"
            f"  resolve_data_root -> {data_root}"
            + (f" (lỗi: {exc_msg})" if exc_msg else f", nhưng {ann} không tồn tại")
            + f"\n  Cần mount dataset chứa {cfg_data['annotation_rel']}.\n"
            f"  Ứng viên khai trong configs/data.yaml: {cfg_data.get('data_root_candidates')}"
        ) from None
    print("data root:", data_root, "✓")

    os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
    rc = subprocess.run(
        [sys.executable, "-m", "src.preprocess.build_cache",
         "--config", "configs/preprocess_e4.yaml"],
        cwd=REPO,
    ).returncode
    assert rc == 0, "build cache thất bại"
    print("build xong:", CACHE_DIR)
else:
    print("bỏ qua build — đã có cache E4:", CACHE_DIR)

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

## Cổng A ⚠️ — cache có đúng là E4 không

In [ ]:
import json

meta = json.loads((Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json").read_text("utf-8"))
for key, want in E4_KEYS.items():
    got = meta.get(key)
    assert got == want, f"cache SAI: {key} = {got!r}, cần {want!r}. Đây không phải cache E4."
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(Path(os.environ["LLDMMRI_CACHE_DIR"]).glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498 — cache chưa build xong"
print(f"cache_meta khớp E4 ✓ · {n_npz} ca · commit {meta.get('git_commit')}")

## 2. Ngân sách

In [ ]:
import time

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "—")

SECONDS_PER_EPOCH = 45.0      # đo thật trên E4: 44.5–45.1 s/epoch qua 5 fold
EPOCHS = int(CFG["train"]["epochs"])
est_hours = SECONDS_PER_EPOCH * EPOCHS / 3600
print(f"\nước tính: {est_hours:.2f} h/fold × {len(FOLDS)} fold = {est_hours * len(FOLDS):.2f} h")
if BUILD_NEEDED:
    print("cộng ~0.45h build cache")
assert est_hours * len(FOLDS) < 11.0, "quá sát trần 12h — bớt fold trong FOLDS"

## 3. Train

Chạy lại **chính cell này** nếu session bị ngắt: `resume: true` nạp lại `last.pt`, và
fold nào đã đủ epoch thì trả về ngay.

In [ ]:
from src.train.run import train

results = {}
t0 = time.time()

for fold in FOLDS:
    used = (time.time() - t0) / 3600
    if used + est_hours > 11.0:
        print(f"\nDỪNG: đã dùng {used:.2f}h, chạy tiếp fold {fold} sẽ vượt trần 12h.")
        print("Fold còn lại để session sau — checkpoint đã lưu, không mất gì.")
        break
    print(f"\n{'=' * 70}\nFOLD {fold}  (đã dùng {used:.2f}h)\n{'=' * 70}")
    results[fold] = train(CFG_PATH, fold_override=fold)
    print(f"fold {fold} xong: macro-F1 {results[fold].get('macro_f1', float('nan')):.4f}")

print(f"\ntổng: {(time.time() - t0) / 3600:.2f} h")

## 4. Kết quả session này

⚠️ Con số dùng để báo cáo là bản **gộp out-of-fold** trên đủ 5 fold, tính ở máy local
bằng `python -m src.eval.run`. Trung bình các fold trong bảng dưới **không** có CI đúng
nghĩa vì mỗi fold là một tập nhỏ khác nhau.

In [ ]:
import numpy as np

OUT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
rows = []
for d in sorted(OUT.glob("fold*")):
    f = d / "metrics_best.json"
    if f.exists():
        m = json.loads(f.read_text("utf-8"))
        rows.append((d.name, m["fold"], m["epoch"], m["macro_f1"], m["cohen_kappa"]))

print(f"{'thư mục':<22}{'fold':>5}{'epoch':>7}{'macro-F1':>11}{'kappa':>9}")
print("-" * 54)
for name, fold, ep, f1, kp in rows:
    print(f"{name:<22}{fold:>5}{ep:>7}{f1:>11.4f}{kp:>9.4f}")

print("\nĐỐI CHIẾU E4 (cross-entropy, cùng cache/seed/split):")
E4 = {1: 0.7001, 2: 0.6771, 3: 0.7304, 4: 0.6680, 5: 0.6618}
for name, fold, ep, f1, kp in rows:
    if fold in E4:
        print(f"  fold {fold}: {f1:.4f} so với {E4[fold]:.4f}  ({f1 - E4[fold]:+.4f})")
print("\n⚠ Chênh lệch từng fold là NHIỄU nếu nhìn riêng lẻ (CI mỗi fold rộng ~0.19).")
print("  Đừng kết luận gì trước khi đủ 5 fold và gộp out-of-fold.")

## 5. Gói mang về

Ở máy local, đặt vào `runs/E5_focal/fold_N/` rồi:

```
python -m src.eval.run   --run-dir runs/E5_focal
python -m src.eval.trust --run-dir runs/E5_focal
```

Bảng thứ hai trả lời **H2** (calibration) — nhớ so ECE cùng trạng thái hiệu chỉnh.

In [ ]:
import shutil

PACK = Path(f"/kaggle/working/{EXPERIMENT}_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

KEEP = ["val_probs_best.npz", "val_probs_last.npz", "metrics_best.json",
        "train_log.csv", "config_used.json", "best.pt"]
for d in sorted(OUT.glob("fold*")):
    dst = PACK / d.name
    dst.mkdir(parents=True, exist_ok=True)
    for name in KEEP:
        if (d / name).exists():
            shutil.copy2(d / name, dst / name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.1f} MiB")
for f in sorted(PACK.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(PACK)}  {f.stat().st_size / 2**20:.2f} MiB")

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz và .pt bản thân là zip; trình giải nén bung
  đệ quy sẽ biến chúng thành thư mục và src.eval.* sẽ không thấy (đã dính, S-078).
""")